# PathSnap — Logistic Regression (Repository-Connected VS Code Notebook)

This notebook reads only `results/features/train/`. It creates binary road-candidate labels using the repository implementation, checks label and trip coverage, runs five-fold `GroupKFold` by trip ID, and fits the final model on all training trips. Student-test and evaluator-test data are not used.

If the kernel starts outside the repository, the first code cell asks for the repository folder. It does not assume a username or Downloads location.

In [ ]:
from pathlib import Path
import os
import sys

def is_pathsnap_repository(path: Path) -> bool:
    return (
        (path / "results" / "features" / "train").is_dir()
        and (path / "src" / "candidate_training.py").is_file()
    )

def find_repository(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if is_pathsnap_repository(candidate):
            return candidate
    return None

PROJECT_ROOT = find_repository(Path.cwd())
if PROJECT_ROOT is None:
    configured_root = os.environ.get("PATHSNAP_REPO_ROOT", "").strip()
    if not configured_root:
        configured_root = input(
            "Repository folder not found from the notebook kernel. "
            "Enter the machine-learning-hackathon folder path: "
        ).strip()
    PROJECT_ROOT = Path(configured_root).expanduser().resolve()
    if not is_pathsnap_repository(PROJECT_ROOT):
        raise FileNotFoundError(
            f"{PROJECT_ROOT} is not the PathSnap repository: expected "
            "results/features/train/ and src/candidate_training.py."
        )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

TRAIN_DIR = PROJECT_ROOT / "results" / "features" / "train"
print("Repository:", PROJECT_ROOT)
print("Training folder:", TRAIN_DIR)
print("Training trip folders:", sum(path.is_dir() for path in TRAIN_DIR.iterdir()))

In [ ]:
from src.notebook_training import run_notebook_training

OUTPUTS = {
    "model_path": PROJECT_ROOT / "models" / "logistic_regression_candidate.joblib",
    "candidates_path": PROJECT_ROOT / "results" / "train_candidates_generated.parquet",
    "cv_results_path": PROJECT_ROOT / "results" / "logistic_regression_cv_results.csv",
    "oof_predictions_path": PROJECT_ROOT / "results" / "logistic_regression_oof_predictions.csv",
    "summary_path": PROJECT_ROOT / "results" / "logistic_regression_training_summary_notebook.json",
}
for name, path in OUTPUTS.items():
    print(f"{name}: {path}")

## Generate candidates and train

Candidate search uses the training-derived settings recorded in the summary: 128 nearest arcs within 200 m, a 70 m/s route-progress ceiling, and route endpoint anchors validated against training-trip endpoints. Observations without a positive reference-route candidate are excluded from supervised rows and counted in candidate recall rather than incorrectly treating every candidate as negative. Preprocessing is fitted independently inside each cross-validation fold.

In [ ]:
summary = run_notebook_training(
    TRAIN_DIR,
    **OUTPUTS,
    top_k=128,
    max_distance_m=200.0,
    cell_size_m=100.0,
    max_route_speed_mps=70.0,
    start_tolerance_m=200.0,
    end_tolerance_m=600.0,
    advance_penalty_m=5.0,
    n_splits=5,
    random_seed=42,
    expected_trip_count=80,
)

print("Training trips:", summary["training_trip_count"])
print("Candidate rows:", summary["candidate_rows"])
print("Positive candidates:", summary["positive_candidates"])
print("Negative candidates:", summary["negative_candidates"])
print("Candidate recall (%):", summary["candidate_recall_pct_all_training_trips"])
print("Pooled out-of-fold metrics:", summary["validation"]["metrics_on_pooled_out_of_fold_predictions"])
print("Files:", summary["outputs"])